# Test LLM-Chat using ./wheels/langchain_ai_portal Wheel

In [1]:
%reload_ext autoreload
%autoreload 2

## Imports & Setup

In [2]:
from dotenv import load_dotenv
import os
import json
import warnings

try:
    from langchain_ai_portal import UbitecAiPortalSdkChat
except ModuleNotFoundError:
    warnings.warn(message="Ai-Portal Package not available.", category=Warning)

# Load environment variables from .env file
load_dotenv()

True

## Test Portal API Langchain Chat Integration

In [ ]:
print("Loading LLM from AI-PORTAL API")
chat_model = UbitecAiPortalSdkChat(
    portal_api_host=os.getenv("PORTAL_API_HOST"),
    portal_api_port=os.getenv("PORTAL_API_PORT"),
    portal_api_sdk_apikey=os.getenv("PORTAL_API_KEY"),
    portal_api_use_ssl=os.getenv("PORTAL_API_USE_SSL"),
    model=os.getenv("PORTAL_API_MODEL_IDENTIFIER"),
    model_kwargs={"temperature": 0.0},
    streaming=True,
)

print(f"Loaded Model {chat_model.model_name} from {chat_model.portal_api_host}")

Loading LLM from AI-PORTAL API
Loaded Model microsoft/phi-4 from dev1.pc1.kipitz.de


In [4]:
system_message = "You are a data privacy expert."

prompt = """Analyze the following text for sensitive information according to GDPR and German 
Informationsfreiheitsgesetz (IFG). Identify any:
1. Personal data (names, addresses, contact info)
2. Special category data (health, religion, political opinions)
3. Official secrets
4. Business secrets or confidential information

Return the results in the following JSON format:
{
    "sensitive_sections": [
        {
            "text": "sensitive text excerpt",
            "category": "category name",
            "reason": "explanation why this is sensitive"
        }
    ]
}
Just give back the JSON and nothing else. Don't add any comments or explanations. Don't say "Here is the JSON" or anything like that.
Text to analyze:
"""

In [5]:
async def call_structured_completion_model(model: UbitecAiPortalSdkChat, user_message):
    message = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt + user_message}
    ]
    response = await model.ainvoke(message)

    return response


user_message = """Vertrauliches Dokument
Kundenname: Max Mustermann
Geburtsdatum: 12.03.1985
Personalausweisnummer: 1234567890
Bankverbindung:
IBAN: DE44 1234 5678 9000 1234 56
BIC: GENODEF1S12
Medizinische Daten:
Diagnose: Bluthochdruck
Behandlung: Lisinopril 10 mg täglich
Vertrauliche Notiz:
Dieses Dokument enthält personenbezogene und sensible Informationen, die
gemäß DSGVO geschützt sind. Unbefugter Zugriff ist nicht gestattet.
--- Interne Verwendung nur ---."""

response = await call_structured_completion_model(chat_model, user_message)
print(f"Response:\n\n\"{response.content}\"")


Response:

"```json
{
    "sensitive_sections": [
        {
            "text": "Kundenname: Max Mustermann",
            "category": "Personal data",
            "reason": "Contains a name, which is considered personal data under GDPR."
        },
        {
            "text": "Geburtsdatum: 12.03.1985",
            "category": "Personal data",
            "reason": "Contains a date of birth, which is considered personal data under GDPR."
        },
        {
            "text": "Personalausweisnummer: 1234567890",
            "category": "Personal data",
            "reason": "Contains an identification number, which is considered personal data under GDPR."
        },
        {
            "text": "Bankverbindung:\nIBAN: DE44 1234 5678 9000 1234 56\nBIC: GENODEF1S12",
            "category": "Personal data",
            "reason": "Contains financial information, which is considered personal data under GDPR."
        },
        {
            "text": "Medizinische Daten:\nDiagnose: Blu

## Naive Extracting and Parsing the Response JSON
Sometimes the response is not valid JSON, but a markdown-formatted string, e.g.: "```json\n{"key": "value"}```"

1. Extract the JSON part
2. Parse the JSON

In [6]:
import re

def extract_json_from_raw_response(raw_response: str) -> dict:
    # Use regex to greedily match from the first '{' to the last '}' in the string
    match = re.search(r'\{.*\}', raw_response, re.DOTALL)
    
    if match:
        json_response = match.group(0) # this is the actual JSON part
        data = json.loads(json_response)
        return data
    else:
        raise ValueError("No valid JSON found in the response.")
    
data = extract_json_from_raw_response(response.content)
print(f"Extracted JSON:\n\n{data}")

Extracted JSON:

{'sensitive_sections': [{'text': 'Kundenname: Max Mustermann', 'category': 'Personal data', 'reason': 'Contains a name, which is considered personal data under GDPR.'}, {'text': 'Geburtsdatum: 12.03.1985', 'category': 'Personal data', 'reason': 'Contains a date of birth, which is considered personal data under GDPR.'}, {'text': 'Personalausweisnummer: 1234567890', 'category': 'Personal data', 'reason': 'Contains an identification number, which is considered personal data under GDPR.'}, {'text': 'Bankverbindung:\nIBAN: DE44 1234 5678 9000 1234 56\nBIC: GENODEF1S12', 'category': 'Personal data', 'reason': 'Contains financial information, which is considered personal data under GDPR.'}, {'text': 'Medizinische Daten:\nDiagnose: Bluthochdruck\nBehandlung: Lisinopril 10 mg täglich', 'category': 'Special category data', 'reason': 'Contains health information, which is considered special category data under GDPR.'}, {'text': 'Dieses Dokument enthält personenbezogene und sensib

In [7]:
data["sensitive_sections"][0]

{'text': 'Kundenname: Max Mustermann',
 'category': 'Personal data',
 'reason': 'Contains a name, which is considered personal data under GDPR.'}